<a href="https://colab.research.google.com/github/erdilix/nma-compneu2026/blob/main/projects/behavior_and_theory/earth_laquitaine_circular_gen_simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rung 3 — add the **circle** (von Mises, generative)

Ladder position: this is the step that changes **exactly one thing** — geometry. `earth_laquitaine_gen_ext.ipynb` built the generative observer on a straight line with Gaussians; here we keep it *generative* and *simple* (one fixed prior) but move onto the **circle**, where directions wrap (359° ≈ 1°).

1. `earth_laquitaine_gen_ext.ipynb` — Gaussian, generative *(done)*
2. `earth_laquitaine_fit_simple.ipynb` — Gaussian, fitted *(done)*
3. **this notebook** — von Mises, generative *(the circle, in isolation)*
4. `earth_laquitaine_circular_fit_ext.ipynb` — von Mises, fitted *(done)*

**The one swap.** Every Gaussian $\mathcal N(\mu,\sigma)$ becomes a von Mises $\mathrm{VM}(\mu,\kappa)$ with concentration $\kappa \approx 1/\sigma^2$, and every mean/difference becomes its circular version. Nothing else about the Bayesian logic changes: still `posterior ∝ prior × likelihood`, still read the MAP, still watch the estimate get pulled toward the 225° prior — but now it behaves correctly across the wrap.

In [ ]:
# @title Dependencies
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams['figure.figsize'] = [8, 4]
rcParams['font.size'] = 11
rcParams['axes.spines.top'] = False
rcParams['axes.spines.right'] = False

MU0 = 225.0   # prior mean, fixed by the experiment

## Step 0 — circular state space + a von Mises on a grid

The state space is the same direction grid, but now we *treat it as a circle*. `von_mises(x, mu, kappa)` is the circular twin of `my_gaussian` from the linear notebook: a bump centred on `mu`, normalised to sum to 1 on the grid. Two circular helpers replace the linear ones:

- `circular_mean` — average of angles (via unit vectors), so 350° and 10° average to 0°, not 180°.
- `signed_diff` — smallest signed angular gap, always in [-180, 180].

In [ ]:
# @title Circular grid + helpers
GRID_STEP = 2
xg = np.arange(1, 361, GRID_STEP)   # direction grid, treated as a circle
n = len(xg)

def von_mises(x, mu, kappa):
    """von Mises bump over the grid x, centred on mu (deg), normalised to sum 1."""
    v = np.exp(kappa * np.cos(np.deg2rad(x - mu)))
    return v / v.sum()

def circular_mean(deg, weights=None):
    r = np.deg2rad(deg)
    c = np.average(np.cos(r), weights=weights)
    s = np.average(np.sin(r), weights=weights)
    return np.degrees(np.arctan2(s, c)) % 360

def signed_diff(target, current):
    """smallest signed angular difference target - current (deg), in [-180, 180]."""
    return ((target - current + 180) % 360) - 180

# kappa ~ 1/sigma^2 : bigger kappa = sharper bump
for kappa in [1, 4, 16]:
    p = von_mises(xg, 90, kappa)
    print(f"kappa={kappa:>2}  (sigma~{1/np.sqrt(kappa):.2f} rad)  peak height {p.max():.3f}")

## Step 1 — the prior

One von Mises centred on 225°. `kappa_prior` is the confidence knob (bigger = narrower = stronger pull). Unlike the linear Gaussian, this bump wraps: put the prior near 0° and it stays smooth across the 360/0 seam.

In [ ]:
# @title The prior, and proof that it wraps
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

for kp in [1, 2, 8]:
    ax[0].plot(xg, von_mises(xg, MU0, kp), label=f"kappa_prior={kp}")
ax[0].axvline(MU0, color="gray", ls=":")
ax[0].set_title("prior = von Mises at 225 deg")
ax[0].set_xlabel("direction (deg)"); ax[0].set_ylabel("probability"); ax[0].legend()

# a prior centred near the seam stays smooth (a linear Gaussian would be chopped)
ax[1].plot(xg, von_mises(xg, 5, 4), label="prior at 5 deg")
ax[1].set_title("wrap works: mass continues past 360 -> 0")
ax[1].set_xlabel("direction (deg)"); ax[1].legend()
plt.tight_layout(); plt.show()

## Step 2 — the likelihood

On each trial the observer gets a noisy measurement $m$ of the true direction. Its reliability is set by the motion **coherence**: high coherence = sharp likelihood (large `kappa_llh`), low coherence = broad. Here we just pick a measurement and a `kappa_llh` to see the shape.

In [ ]:
# @title The likelihood for one measurement
meas_deg = 160.0
plt.figure()
for kllh, lab in [(1.0, "low coherence"), (4.0, "mid"), (16.0, "high coherence")]:
    plt.plot(xg, von_mises(xg, meas_deg, kllh), label=f"{lab} (kappa_llh={kllh})")
plt.axvline(meas_deg, color="k", ls="--", label=f"measurement {meas_deg:.0f} deg")
plt.title("likelihood: higher coherence -> sharper")
plt.xlabel("direction (deg)"); plt.ylabel("probability"); plt.legend()
plt.tight_layout(); plt.show()

## Step 3 — Bayes: posterior = prior × likelihood

Multiply the two bumps point-by-point on the grid and renormalise. On the circle there is a neat fact: the product of two von Mises distributions is *again* a von Mises, whose centre is the **phasor (vector) sum** of the two — the circular version of the precision-weighted average. So the posterior peak sits between the measurement and 225°, and its concentration is roughly $\kappa_{\text{llh}} + \kappa_{\text{prior}}$.

In [ ]:
# @title Posterior and the MAP readout
kp = 2.0
prior = von_mises(xg, MU0, kp)

plt.figure()
plt.plot(xg, prior, "--", color=[0.75, 0.75, 0], label="prior (225)")
for kllh, col in [(1.5, "tab:blue"), (16.0, "tab:red")]:
    like = von_mises(xg, meas_deg, kllh)
    post = prior * like; post = post / post.sum()   # Bayes
    est = xg[post.argmax()]                          # MAP
    plt.plot(xg, like, ":", color=col, alpha=0.6)
    plt.plot(xg, post, "-", color=col, label=f"posterior kappa_llh={kllh} -> MAP {est:.0f} deg")
    plt.axvline(est, color=col, alpha=0.4)
plt.axvline(meas_deg, color="gray", ls=":"); plt.axvline(MU0, color="gray", ls=":")
plt.title("low coherence (blue) is pulled harder toward 225")
plt.xlabel("direction (deg)"); plt.ylabel("probability"); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Step 4 — the observer as a function

Wrap Steps 2-3 into one call: given a measurement and the two concentrations, return the MAP estimate. This *is* the generative observer — feed it a measurement, get a report.

In [ ]:
# @title estimate(measurement) -> MAP direction
def estimate(meas_deg, kappa_llh, kappa_prior, prior_mean=MU0):
    prior = von_mises(xg, prior_mean, kappa_prior)
    like = von_mises(xg, meas_deg, kappa_llh)
    post = prior * like
    return xg[post.argmax()]

for m in [160, 300, 10]:
    e = estimate(m, kappa_llh=2.0, kappa_prior=2.0)
    print(f"measurement {m:>3} deg -> estimate {e:>3.0f} deg "
          f"(shift toward 225: {signed_diff(e, m):+.0f} deg)")

## Step 5 — sweep: the coherence-dependent bias

Now run it generatively across the experiment. For each true stimulus and each coherence, draw many noisy measurements from a von Mises around the stimulus, push each through `estimate`, and take the **circular** average of the reports. The classic result: estimates bow away from veridical toward 225°, and the bow is largest at low coherence.

In [ ]:
# @title Simulate mean estimate vs stimulus, per coherence
rng = np.random.default_rng(0)
stimuli = np.arange(160, 291, 15)          # true directions spanning around the prior
kappa_by_coh = [(1.5, "low coh"), (4.0, "mid coh"), (16.0, "high coh")]
N_SAMPLES = 400

plt.figure()
for (kllh, lab), col in zip(kappa_by_coh, ["tab:blue", "tab:green", "tab:red"]):
    mean_est = []
    for th in stimuli:
        meas = np.degrees(rng.vonmises(np.deg2rad(th), kllh, size=N_SAMPLES)) % 360
        ests = np.array([estimate(m, kllh, kappa_prior=2.0) for m in meas])
        mean_est.append(circular_mean(ests))
    plt.plot(stimuli, mean_est, "-o", color=col, label=lab)
plt.plot(stimuli, stimuli, "r--", lw=1, label="veridical")
plt.axhline(MU0, color="gray", ls=":", label="prior 225")
plt.title("estimate bows toward 225, most at low coherence")
plt.xlabel("stimulus direction (deg)"); plt.ylabel("mean estimate (deg)")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

## What the circle changed, and what's next

- **Same Bayesian logic.** prior × likelihood → normalise → MAP → bias toward the prior, growing as coherence falls. Identical story to `earth_laquitaine_gen_ext.ipynb`.
- **What actually changed.** Gaussian → von Mises ($\kappa \approx 1/\sigma^2$), plain mean/difference → circular mean / signed angular difference. The payoff: everything stays correct near the 0/360 seam, where the linear model silently breaks.
- **Rung 4 — fit it to real data.** Swap "draw measurements and produce estimates" for "score the subject's actual estimates and minimise NLL". Because the circular MAP has no simple closed form, that step marginalises the hidden measurement on the grid — exactly `earth_laquitaine_circular_fit_ext.ipynb`.
- **Still open (optional slots):** `circular_gen_ext` (add online / warm-up priors, generative) and `circular_fit_simple` (static-only circular fit) would fill the last two cells of the 8-slot grid.